In [ ]:
!pip install -q faster-whisper librosa soundfile jiwer

import time
import librosa
import soundfile as sf
import numpy as np
from faster_whisper import WhisperModel
from jiwer import wer, cer

AUDIO_FILE = "out-0015592831-5053-20260802-125454-1785654594.80151.wav"

MODEL_SIZE = "large-v3"

RESAMPLED_FILE = "resampled_16k.wav"

OUTPUT_TEXT_FILE = "transcription_vad_off.txt"



REFERENCE_TEXT = """हेल्लो हजुर सुब्बाकार्यबाट बोल्नुभएको हो नि म्याम हैन हजुर हजुर एकछिन मैले अहिले चलाउँदै थिएँ एकछिन एक्सेप्ट गर्दिनु न है मेरो काम सकिएको छैन त्यही भएर ए एक्सेप्ट गरेँ अनि रोक्नुस् है हजुरले अहिलेको इयरमा नि चेक गर्नुभयो हैन हजुर हजुर एकचोटि लास्ट इयरमा लगिन गरेर मलाई देखाउनुस् न है लग अफ गर्ने लगिन लास्ट इयरमा हजुरको लास्ट इयरमा नि हजुर लास्ट इयरको लगिन गरेर देखाउनुस् न है यहाँबाट होला सायद विन्डोबाट हजुर लगिन अ त्यहाँबाट अब लास्ट इयरको लगिन गर्ने है अब ड्रप डाउन गर्नु सरले यही देखाउनुभएको थियो मलाई त्यही भएर एकचोटि तल टिक लगाउनु न यतापट्टि यतापट्टि ल फेरि कहाँ गयो अघि यसैमा गरेको हैन र अघि यसैमा गरेको हो अब हुनुपर्ने हो रजनीगन्धा देऊ त यसैमा हैन हजुर अब सेकेन्डमा जाने हजुर अब त्यहाँ ड्रप डाउनमा जाने अब ८२ ८३ देखाएको छैन है यो दुइटा २३० + २६५ एकछिन है म चेक गर्छु ल ल म होल्डमा राख्नु हो अझ तल तल गर्नु न देखाएको होला देखाएको छ नि म्याम अब ८२ ८३ मा जानु अनि लगिन गर्नु अब लगिन गर्नु लगिन सक्सेसफुल भयो है मलाई एकछिन है मैले यो एसक्यूएल बन्द गर्छु एकैछिन है अब अरु केही प्रोब्लम आयो भने कल गर्नु न है ओके हस् थ्याङ्क यू पख्नुस् एकैछिन है म एसक्यूएल बन्द गर्छु यसलाई मैले चलाइरहेको बन्द गर्छु है ए हस् हस् हुन्छ म्याम हस् थ्याङ्क यू हस् वेलकम"""



def preprocess_audio(input_path, output_path, target_sr=16000):

    print("=" * 60)
    print("AUDIO PREPROCESSING")
    print("=" * 60)

    print(f"Input file: {input_path}")

    y, sr = librosa.load(
        input_path,
        sr=None,
        mono=True
    )

    print(f"Original sample rate : {sr} Hz")
    print(f"Original duration    : {len(y) / sr:.2f} seconds")



    if sr != target_sr:

        print(f"Resampling {sr} Hz -> {target_sr} Hz")

        y = librosa.resample(
            y,
            orig_sr=sr,
            target_sr=target_sr
        )

    else:
        print("Audio already at 16 kHz")

    peak = np.max(np.abs(y))

    if peak > 0:

        y = y / peak * 0.95

        print("Peak normalization applied.")

    else:

        print("WARNING: Audio appears to be silent!")

    sf.write(
        output_path,
        y,
        target_sr
    )

    print(f"Saved processed audio: {output_path}")
    print(f"Final sample rate    : {target_sr} Hz")
    print(f"Final duration       : {len(y) / target_sr:.2f} seconds")

    return output_path



def normalize_text(text):
    text = " ".join(text.split())
    punctuation = "।,?!.'\"-–—:;()[]{}"

    for char in punctuation:
        text = text.replace(char, "")

    text = " ".join(text.split())

    return text.strip()



def run_benchmark():

    audio_path = preprocess_audio(
        AUDIO_FILE,
        RESAMPLED_FILE
    )

    print("\n" + "=" * 60)
    print("LOADING WHISPER MODEL")
    print("=" * 60)

    print(f"Model: Faster-Whisper {MODEL_SIZE}")
    print("Device: CUDA")
    print("Compute type: float16")

    model = WhisperModel(
        MODEL_SIZE,
        device="cuda",
        compute_type="float16"
    )

    print("Model loaded successfully.")


    print("\n" + "=" * 60)
    print("TRANSCRIPTION")
    print("=" * 60)

    print("VAD: OFF")
    print("Initial prompt: OFF")
    print("Language: Nepali")
    print("Condition on previous text: OFF")

    start_time = time.time()

    segments, info = model.transcribe(

        audio_path,
        language="ne",

  
        beam_size=5,

        vad_filter=False,

        condition_on_previous_text=False,

        compression_ratio_threshold=2.4,

        log_prob_threshold=-1.5,

        no_speech_threshold=0.6,

        temperature=[
            0.0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0
        ]
    )


    generated_text = ""

    segment_log = []

    prev_end = 0.0

    coverage_seconds = 0.0

    segment_count = 0


    for segment in segments:

        segment_count += 1

        gap = segment.start - prev_end

        if gap > 2.0:

            segment_log.append(
                f"\n*** GAP: {gap:.2f}s "
                f"with NO segment produced "
                f"({prev_end:.2f}s -> {segment.start:.2f}s) ***"
            )

        segment_duration = segment.end - segment.start

        coverage_seconds += segment_duration

        prev_end = segment.end

        generated_text += segment.text.strip() + " "

        segment_log.append(
            f"[{segment.start:7.2f}s -> "
            f"{segment.end:7.2f}s] "
            f"duration={segment_duration:6.2f}s | "
            f"avg_logprob={segment.avg_logprob:6.2f} | "
            f"compression_ratio={segment.compression_ratio:5.2f} | "
            f"no_speech_prob={segment.no_speech_prob:5.2f} | "
            f"{segment.text.strip()}"
        )


    end_time = time.time()



    execution_time = end_time - start_time

    audio_duration = info.duration

    rtf = execution_time / audio_duration


    clean_generated = normalize_text(
        generated_text
    )

    clean_reference = normalize_text(
        REFERENCE_TEXT
    )


    error_rate_wer = wer(
        clean_reference,
        clean_generated
    )

    error_rate_cer = cer(
        clean_reference,
        clean_generated
    )

    cer_accuracy = (
        1 - error_rate_cer
    ) * 100




    coverage_percentage = (
        coverage_seconds /
        audio_duration
    ) * 100



    print("\n")
    print("=" * 70)
    print("NEPALI WHISPER BASELINE PERFORMANCE REPORT")
    print("=" * 70)

    print(
        f"Audio Duration       : "
        f"{audio_duration:.2f} seconds"
    )

    print(
        f"Processing Time      : "
        f"{execution_time:.2f} seconds"
    )

    print(
        f"RTF                  : "
        f"{rtf:.4f}"
    )

    print(
        f"Number of Segments   : "
        f"{segment_count}"
    )

    print(
        f"Word Error Rate      : "
        f"{error_rate_wer * 100:.2f}%"
    )

    print(
        f"Character Error Rate : "
        f"{error_rate_cer * 100:.2f}%"
    )

    print(
        f"CER Accuracy         : "
        f"{cer_accuracy:.2f}%"
    )

    print(
        f"Speech Coverage      : "
        f"{coverage_seconds:.2f}s / "
        f"{audio_duration:.2f}s "
        f"({coverage_percentage:.1f}%)"
    )

    print("=" * 70)



    print("\n")
    print("=" * 70)
    print("SEGMENT DIAGNOSTICS")
    print("=" * 70)

    for line in segment_log:

        print(line)




    print("\n")
    print("=" * 70)
    print("GENERATED TRANSCRIPTION")
    print("=" * 70)

    print(generated_text)


    print("\n")
    print("=" * 70)
    print("REFERENCE TRANSCRIPTION")
    print("=" * 70)

    print(REFERENCE_TEXT)


    with open(
        OUTPUT_TEXT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "NEPALI WHISPER TRANSCRIPTION\n"
        )

        f.write(
            "=" * 60 + "\n\n"
        )

        f.write(
            "Generated Transcription:\n\n"
        )

        f.write(
            generated_text
        )

        f.write(
            "\n\n"
        )

        f.write(
            "Reference Transcription:\n\n"
        )

        f.write(
            REFERENCE_TEXT
        )

        f.write(
            "\n\n"
        )

        f.write(
            "=" * 60 + "\n"
        )

        f.write(
            f"Audio Duration: "
            f"{audio_duration:.2f}s\n"
        )

        f.write(
            f"Processing Time: "
            f"{execution_time:.2f}s\n"
        )

        f.write(
            f"RTF: {rtf:.4f}\n"
        )

        f.write(
            f"WER: "
            f"{error_rate_wer * 100:.2f}%\n"
        )

        f.write(
            f"CER: "
            f"{error_rate_cer * 100:.2f}%\n"
        )

        f.write(
            f"CER Accuracy: "
            f"{cer_accuracy:.2f}%\n"
        )

        f.write(
            f"Speech Coverage: "
            f"{coverage_percentage:.1f}%\n"
        )

        f.write(
            "\n\nSEGMENT DIAGNOSTICS\n\n"
        )

        for line in segment_log:

            f.write(
                line + "\n"
            )


    print("\n")
    print(
        f"Transcription saved to: "
        f"{OUTPUT_TEXT_FILE}"
    )

    return {
        "generated_text": generated_text,
        "reference_text": REFERENCE_TEXT,
        "wer": error_rate_wer,
        "cer": error_rate_cer,
        "cer_accuracy": cer_accuracy,
        "audio_duration": audio_duration,
        "processing_time": execution_time,
        "rtf": rtf,
        "coverage_seconds": coverage_seconds,
        "coverage_percentage": coverage_percentage,
        "segment_count": segment_count
    }



results = run_benchmark()

AUDIO PREPROCESSING
Input file: out-0015592831-5053-20260802-125454-1785654594.80151.wav
Original sample rate : 8000 Hz
Original duration    : 157.10 seconds
Resampling 8000 Hz -> 16000 Hz
Peak normalization applied.
Saved processed audio: resampled_16k.wav
Final sample rate    : 16000 Hz
Final duration       : 157.10 seconds

LOADING WHISPER MODEL
Model: Faster-Whisper large-v3
Device: CUDA
Compute type: float16
Model loaded successfully.

TRANSCRIPTION
VAD: OFF
Initial prompt: OFF
Language: Nepali
Condition on previous text: OFF


NEPALI WHISPER BASELINE PERFORMANCE REPORT
Audio Duration       : 157.10 seconds
Processing Time      : 59.46 seconds
RTF                  : 0.3785
Number of Segments   : 45
Word Error Rate      : 94.25%
Character Error Rate : 58.99%
CER Accuracy         : 41.01%
Speech Coverage      : 137.18s / 157.10s (87.3%)


SEGMENT DIAGNOSTICS
[   0.00s ->   14.40s] duration= 14.40s | avg_logprob= -0.37 | compression_ratio= 2.11 | no_speech_prob= 0.46 | आजु सुबर काईरे